In [190]:
#imports
import spacy
import scispacy
import timeit
import pandas as pd
import os
from pathlib import Path
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer


In [ ]:
vectorizer = CountVectorizer(lowercase=True,)


In [169]:
#load term parser
nlp = spacy.load("en_core_sci_lg")

KeyboardInterrupt: 

In [ ]:
#change this to live file ingest later

file_path = Path.cwd().parent / "archs4metadata/OSD-100|Mmus_C57-6J_EYE_FLT_Rep1_M23_archs4_top10hits_metadata.csv"

md_df = pd.read_csv(file_path)

idx = defaultdict(set) #may want to cache later if it gets big

In [204]:
def _add_spacy_terms(): # adds spacy terms to dataframe, returns list of flat terms
    md_df['spacey_terms'] = md_df.apply(lambda _: [], axis=1)

    for row in md_df.itertuples():
        doc = nlp(row.geo_summary)
        for ent in doc.ents:
            term = ent.text.split()
            row.spacey_terms.extend(term)
            for t in term:
                idx[t].add(row.gse)
            
    
    return [item for lst in md_df["spacey_terms"] for item in lst]

In [213]:
def _update_idx():
    
    analyzer = vectorizer.build_analyzer()
    new_idx = defaultdict(set)

    for term,doc_ids in idx.items():
        tokenized = analyzer(term)
        for token in tokenized:
            new_idx[token].update(doc_ids)

    return new_idx  
            



In [216]:
print(_update_idx())

defaultdict(<class 'set'>, {'female': {'GSE210492'}, 'aged': {'GSE210492'}, 'efemp1': {'GSE210492'}, 'r345w': {'GSE210492'}, 'efemp1ki': {'GSE210492'}, 'ki': {'GSE210492'}, 'mice': {'GSE205070', 'GSE210492', 'GSE124745'}, 'model': {'GSE210492'}, 'doyne': {'GSE210492'}, 'retinal': {'GSE205070', 'GSE210492'}, 'dystrophy': {'GSE210492'}, 'malattia': {'GSE210492'}, 'leventinese': {'GSE210492'}, 'rna': {'GSE210492'}, 'sequencing': {'GSE210492', 'GSE124745'}, 'expression': {'GSE205070', 'GSE210492'}, 'gene': {'GSE205070', 'GSE210492'}, 'set': {'GSE210492'}, 'enrichment': {'GSE210492'}, 'analysis': {'GSE210492'}, 'data': {'GSE210492'}, 'seq': {'GSE210492'}, 'neural': {'GSE210492'}, 'retina': {'GSE143281', 'GSE210492'}, 'posterior': {'GSE210492'}, 'eyecups': {'GSE210492'}, 'whole': {'GSE210492'}, 'eye': {'GSE210492', 'GSE124745', 'GSE88819'}, 'lens': {'GSE210492'}, 'knockout': {'GSE205070', 'GSE124745', 'GSE143281'}, 'ko': {'GSE205070'}, 'mouse': {'GSE205070', 'GSE124745', 'GSE143281'}, 'model

In [217]:
print(idx)

defaultdict(<class 'set'>, {'female': {'GSE210492'}, 'aged': {'GSE210492'}, 'Efemp1': {'GSE210492'}, 'R345W': {'GSE210492'}, 'Efemp1ki/ki': {'GSE210492'}, 'mice': {'GSE205070', 'GSE210492', 'GSE124745'}, 'model': {'GSE210492'}, 'Doyne': {'GSE210492'}, 'retinal': {'GSE205070', 'GSE210492'}, 'dystrophy/malattia': {'GSE210492'}, 'leventinese': {'GSE210492'}, 'RNA': {'GSE210492'}, 'sequencing': {'GSE210492', 'GSE124745'}, 'expression': {'GSE205070', 'GSE210492'}, 'gene': {'GSE205070', 'GSE210492'}, 'set': {'GSE210492'}, 'enrichment': {'GSE210492'}, 'analysis': {'GSE210492'}, 'data': {'GSE210492'}, 'RNA-seq': {'GSE210492'}, 'neural': {'GSE210492'}, 'retina': {'GSE143281', 'GSE210492'}, 'posterior': {'GSE210492'}, 'eyecups': {'GSE210492'}, 'whole': {'GSE210492'}, 'eye': {'GSE210492', 'GSE124745', 'GSE88819'}, 'lens': {'GSE210492'}, 'Efemp1+/+': {'GSE210492'}, 'Knockout': {'GSE205070'}, '(KO)': {'GSE205070'}, 'mouse': {'GSE205070', 'GSE124745', 'GSE143281'}, 'models': {'GSE205070'}, 'biologic

In [227]:
def _add_journals(cl_df):
     cl_df['journals'] = cl_df.apply(lambda _: [], axis=1)
     cl_df["Representation"] = cl_df["Representation"].apply(lambda lst: [x for x in lst if x])

     for row in cl_df.itertuples():
        
        
        for term in row.Representation:
            
            if term in idx:
               row.journals.extend(list(idx[term]))
            else:
                  print(f"term {term} not found in index")
     
     return cl_df      
         

In [228]:

def cluster_sample(): 
    global idx
    flat_terms = _add_spacy_terms()
    vectorizer.fit(flat_terms)
    idx = _update_idx()
    model = SentenceTransformer('allenai/biomed_roberta_base')
    topic_model = BERTopic(vectorizer_model=vectorizer, embedding_model=model)
    topics,probs = topic_model.fit_transform(flat_terms)
    clusters = _add_journals(topic_model.get_topic_info())   
    return topic_model,clusters
    


In [229]:
model,clusters = cluster_sample()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21813.93it/s]
[transformers] RobertaModel LOAD REPORT from: allenai/biomed_roberta_base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [233]:
from collections import Counter

In [241]:
def get_cluster_distribution(cluster_idx):
    journals = clusters.iloc[cluster_idx].journals
    cnts = Counter(journals)
   
    total = len(journals)

    percent_dist =  {name: (count / total) * 100 for name, count in cnts.items()}
    return percent_dist

In [248]:
def get_cluster_counts(cluster_idx):
    journals = clusters.iloc[cluster_idx].journals
    cnts = Counter(journals)
    cnts = dict(cnts)
    return cnts

In [243]:
def get_top_cluster_distribution(n=5):
    top_clusters = clusters.head(n)
    distributions = {}
    for idx, row in top_clusters.iterrows():
        distributions[row.Topic] = get_cluster_distribution(idx)
    return distributions

In [272]:
def get_top_journals(n=5):
    
    top_clusters = clusters.head(n).iloc[1:] #skip outliers
    overall_cnts = defaultdict(int)
    for idx, row in top_clusters.iterrows():
       # print(f"Processing cluster {row.Topic} at index {idx}")
        journal_counts = get_cluster_counts(idx)
       # print(f"Journal counts for cluster {row.Topic}: {journal_counts}")
        for journal, count in journal_counts.items():
            #print(f"Adding {count} to overall count for journal {journal}")
            overall_cnts[journal] += count
    
    return overall_cnts

In [277]:
def get_top_journal_distribution(top_cnt_dict):
    total = sum(top_cnt_dict.values())
    percent_dist = {name: (count / total) * 100 for name, count in top_cnt_dict.items()}
    return percent_dist


In [278]:
get_top_journal_distribution(get_top_journals(5))

{'GSE205070': 48.888888888888886,
 'GSE210492': 22.22222222222222,
 'GSE124745': 13.333333333333334,
 'GSE88819': 6.666666666666667,
 'GSE143281': 8.88888888888889}

In [244]:
get_top_cluster_distribution(5)

{-1: {'GSE205070': 28.57142857142857,
  'GSE124745': 21.428571428571427,
  'GSE143281': 7.142857142857142,
  'GSE210492': 28.57142857142857,
  'GSE88819': 14.285714285714285},
 0: {'GSE205070': 58.333333333333336,
  'GSE210492': 25.0,
  'GSE124745': 8.333333333333332,
  'GSE88819': 8.333333333333332},
 1: {'GSE205070': 54.54545454545454,
  'GSE210492': 27.27272727272727,
  'GSE143281': 18.181818181818183},
 2: {'GSE205070': 18.181818181818183,
  'GSE210492': 27.27272727272727,
  'GSE124745': 27.27272727272727,
  'GSE88819': 9.090909090909092,
  'GSE143281': 18.181818181818183},
 3: {'GSE205070': 63.63636363636363,
  'GSE210492': 9.090909090909092,
  'GSE124745': 18.181818181818183,
  'GSE88819': 9.090909090909092}}